<a href="https://www.kaggle.com/code/garvitsonawala/bnn-modeling?scriptVersionId=273308229" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        file = os.path.join(dirname, filename)
        print(file)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/strength-from-7-to-17-times/pta_dataset.npz
/kaggle/input/final-data-i-hope/pta_dataset.npz
/kaggle/input/psrterm-false-dataset/pta_dataset.npz
/kaggle/input/trial-1-for-consistency/pta_dataset.npz
/kaggle/input/strength1-14/pta_dataset.npz
/kaggle/input/strength1-46-1e91e10/pta_dataset.npz


## 1. Loading of Data

In [2]:
import numpy as np

data = np.load("/kaggle/input/psrterm-false-dataset/pta_dataset.npz")
X_data = data['X']
Y_data = data['Y']
SNRs = data['SNR']

## 2. Looking at range of peak values of the samples

In [3]:
import numpy as np

# Assuming X_data is a 2D NumPy array: shape (n_samples, n_residuals)
peak_to_peak_values = np.ptp(X_data, axis=1)  # axis=1 means row-wise (per sample)

print(peak_to_peak_values)  # This is the list of peak-to-peak amplitudes
print("Min strength:", f"{np.min(peak_to_peak_values):.4e}")
print("Max strength:", f"{np.max(peak_to_peak_values):.4e}")

[5.05716988e-06 2.92224770e-06 5.08866439e-06 ... 3.93114579e-06
 3.44751604e-06 3.04942916e-06]
Min strength: 2.5770e-06
Max strength: 5.7335e-06


## 3. Test Train Splitting and Standardizing

### Test:8% , Train:82.8% , Val:9.2%


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split dataset into train+val and test (keep SNRs aligned)
X_tv, X_test, Y_tv, Y_test, snr_train, snr_test = train_test_split(
    X_data, Y_data, SNRs, test_size=0.08, random_state=42
)

# Split train+val into train and validation
X_train, X_val, Y_train, Y_val = train_test_split(
    X_tv, Y_tv, test_size=0.1, random_state=42
)

# --- Standardize X ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)  

# Reshape Y
Y_train = Y_train.reshape(-1, 1)
Y_val = Y_val.reshape(-1, 1)
Y_test = Y_test.reshape(-1, 1)

# --- Standardize Y ---
scaler_y = StandardScaler()
Y_train_scaled = scaler_y.fit_transform(Y_train)
Y_val_scaled = scaler_y.transform(Y_val)
Y_test_scaled = scaler_y.transform(Y_test)


In [5]:
expected_sigma = np.sqrt(np.mean((Y_test_scaled / snr_test) ** 2))
print(expected_sigma)

0.08301139928669195


## 4. Converting the Datasets to tensor

In [6]:
import torch
X_train_tensor = torch.from_numpy(X_train_scaled).float()
X_val_tensor = torch.from_numpy(X_val_scaled).float()
X_test_tensor = torch.from_numpy(X_test_scaled).float()
Y_train_tensor = torch.from_numpy(Y_train_scaled).float()
Y_val_tensor = torch.from_numpy(Y_val_scaled).float()
Y_test_tensor = torch.from_numpy(Y_test_scaled).float()

## 5. Looking at the Structure of the training set

In [7]:
print(type(X_train_tensor))
print(X_train_tensor.shape)
print(Y_train_tensor.shape)

<class 'torch.Tensor'>
torch.Size([12420, 1179])
torch.Size([12420, 1])


## 6. Installing Pyrotorch

In [8]:
pip install pyro-ppl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 69.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found existing installation: nvidia-curand-cu12 10.3.10.19
    Uninstalling nvidia-curand-cu12-

In [9]:
import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.nn import PyroModule, PyroSample
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from pyro.infer.autoguide import AutoLowRankMultivariateNormal


class BayesianNN(PyroModule):
    def __init__(self, input_dim, hidden_size, prior_scale, expected_sigma, dropout):
        super().__init__()
        self.hidden_size = hidden_size
        self.prior_scale = prior_scale

        # Layer 1
        self.fc1 = PyroModule[nn.Linear](input_dim, hidden_size)
        self.fc1.weight = PyroSample(
            dist.Normal(0., prior_scale).expand([hidden_size, input_dim]).to_event(2)
        )
        self.fc1.bias = PyroSample(
            dist.Normal(0., prior_scale).expand([hidden_size]).to_event(1)
        )
        self.bn1 = nn.BatchNorm1d(hidden_size)

        # Layer 2
        self.fc2 = PyroModule[nn.Linear](hidden_size, hidden_size)
        self.fc2.weight = PyroSample(
            dist.Normal(0., prior_scale).expand([hidden_size, hidden_size]).to_event(2)
        )
        self.fc2.bias = PyroSample(
            dist.Normal(0., prior_scale).expand([hidden_size]).to_event(1)
        )
        self.bn2 = nn.BatchNorm1d(hidden_size)

        # Output layer
        self.out_mean = PyroModule[nn.Linear](hidden_size, 1)
        self.out_mean.weight = PyroSample(
            dist.Normal(0., prior_scale).expand([1, hidden_size]).to_event(2)
        )
        self.out_mean.bias = PyroSample(
            dist.Normal(0., prior_scale).expand([1]).to_event(1)
        )

        # Homoscedastic observation noise prior
        self.obs_sigma = PyroSample(dist.HalfNormal(expected_sigma))

        # Other modules
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, y=None):
        x = self.dropout(self.relu(self.bn1(self.fc1(x))))
        x = self.dropout(self.relu(self.bn2(self.fc2(x))))

        mean = self.out_mean(x).squeeze(-1)

        # Ensure positivity of sigma
        sigma = torch.clamp(self.obs_sigma, min=1e-6)
        sigma_expanded = sigma.expand(x.shape[0])

        with pyro.plate("data", x.shape[0]):
            obs = pyro.sample("obs", dist.Normal(mean, sigma_expanded), obs=y)
        return mean


class BNNTrainer:
    def __init__(self, input_dim, hidden_size=30, prior_scale=0.5,
                 expected_sigma=1.0, dropout=0.2, lr=5e-4, clip_norm=10.0):
        self.model = BayesianNN(
            input_dim=input_dim,
            hidden_size=hidden_size,
            prior_scale=prior_scale,
            expected_sigma=expected_sigma,
            dropout=dropout
        )
        self.guide = AutoLowRankMultivariateNormal(self.model_fn)
        self.lr = lr
        self.clip_norm = clip_norm

    def model_fn(self, x, y=None):
        return self.model(x, y)

    def train(self, x, y, num_epochs=900):
        pyro.clear_param_store()
        optimizer = ClippedAdam({"lr": self.lr, "clip_norm": self.clip_norm})
        svi = SVI(self.model_fn, self.guide, optimizer, loss=Trace_ELBO())

        for epoch in range(num_epochs):
            loss = svi.step(x, y)
            if epoch % 100 == 0:
                print(f"Epoch {epoch} | Loss = {loss:.4f}")
        return self.model


In [10]:
"""# --- Hyperparameters to tune --- First run
    hidden_size = trial.suggest_int("hidden_size", 32, 64)
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    prior_scale = trial.suggest_float("prior_scale", 0.1, 1.0)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    expected_sigma = trial.suggest_float("expected_sigma", 0.1, 2.0)
# --- Hyperparameters to tune more NARROW --- Second run
    hidden_size = trial.suggest_int("hidden_size", 32, 64)   # narrow around 44
    lr = trial.suggest_float("lr", 5e-5, 2e-3, log=True)     # best was ~9e-4
    prior_scale = trial.suggest_float("prior_scale", 0.05, 0.3)   # good ones 0.1–0.2
    dropout = trial.suggest_float("dropout", 0.15, 0.4)      # best cluster 0.25–0.35
    expected_sigma = trial.suggest_float("expected_sigma", 0.3, 1.0)  # best ~0.6
# --- Hyperparameters (narrowed search space) ---
    hidden_size = trial.suggest_int("hidden_size", 50, 64)   # best ~60
    lr = trial.suggest_float("lr", 1e-4, 6e-4, log=True)     # best ~2.4e-4
    prior_scale = trial.suggest_float("prior_scale", 0.07, 0.12)  # best ~0.08–0.1
    dropout = trial.suggest_float("dropout", 0.35, 0.45)     # best ~0.39
    expected_sigma = trial.suggest_float("expected_sigma", 0.4, 0.6)  # HalfNormal scale"""


'# --- Hyperparameters to tune --- First run\n    hidden_size = trial.suggest_int("hidden_size", 32, 64)\n    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)\n    prior_scale = trial.suggest_float("prior_scale", 0.1, 1.0)\n    dropout = trial.suggest_float("dropout", 0.0, 0.5)\n    expected_sigma = trial.suggest_float("expected_sigma", 0.1, 2.0)\n# --- Hyperparameters to tune more NARROW --- Second run\n    hidden_size = trial.suggest_int("hidden_size", 32, 64)   # narrow around 44\n    lr = trial.suggest_float("lr", 5e-5, 2e-3, log=True)     # best was ~9e-4\n    prior_scale = trial.suggest_float("prior_scale", 0.05, 0.3)   # good ones 0.1–0.2\n    dropout = trial.suggest_float("dropout", 0.15, 0.4)      # best cluster 0.25–0.35\n    expected_sigma = trial.suggest_float("expected_sigma", 0.3, 1.0)  # best ~0.6\n# --- Hyperparameters (narrowed search space) ---\n    hidden_size = trial.suggest_int("hidden_size", 50, 64)   # best ~60\n    lr = trial.suggest_float("lr", 1e-4, 6e-4, 

In [11]:
import os
import json
import datetime
import optuna
import torch.nn.functional as F

# --- Your existing objective (unchanged) ---
def objective(trial):
    hidden_size = trial.suggest_int("hidden_size", 32, 64)
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    prior_scale = trial.suggest_float("prior_scale", 0.05, 0.4)
    dropout = trial.suggest_float("dropout", 0.1, 0.44)
    expected_sigma = trial.suggest_float("expected_sigma", 0.1, 0.6)

    alpha, beta = 0.5, 0.5

    trainer = BNNTrainer(
        input_dim=X_train_tensor.shape[1],
        hidden_size=hidden_size,
        prior_scale=prior_scale,
        expected_sigma=expected_sigma,
        dropout=dropout,
        lr=lr
    )

    trainer.train(
        X_train_tensor,
        Y_train_tensor.squeeze(),
        num_epochs=900
    )

    with torch.no_grad():
        preds = trainer.model(X_val_tensor)

        mse = F.mse_loss(preds, Y_val_tensor.squeeze()).item()

        sigma = torch.clamp(trainer.model.obs_sigma, min=1e-6)
        nll = -torch.distributions.Normal(preds, sigma).log_prob(
            Y_val_tensor.squeeze()
        ).mean().item()

    return alpha * mse + beta * nll


# --- Storage + study setup (persistent) ---
storage = "sqlite:///bnn_optuna.db"
study_name = "bnn_study"

# Use TPE sampler by default (recommended). You can also tune sampler params.
sampler = optuna.samplers.TPESampler()

pruner = optuna.pruners.MedianPruner(n_warmup_steps=50)

study = optuna.create_study(
    study_name=study_name,
    direction="minimize",
    storage=storage,
    load_if_exists=True,
    sampler=sampler,
    pruner=pruner
)

# --- CSV logging callback (appends per-trial) ---
csv_logfile = "bnn_optuna_trials.csv"

def trial_logger(study, trial):
    # extract the finished trial info
    rec = {
        "datetime": datetime.datetime.now().isoformat(),
        "trial_number": trial.number,
        "state": str(trial.state),
        "value": float(trial.value) if trial.value is not None else None
    }
    # add all params into the record
    for k, v in trial.params.items():
        rec[k] = v

    # append to csv (create header if not exists)
    df = pd.DataFrame([rec])
    header = not os.path.exists(csv_logfile)
    df.to_csv(csv_logfile, mode="a", header=header, index=False)


# --- Run optimization; results will be appended to the SQLite DB ---
# Pass the callback so each finished trial is logged to CSV
study.optimize(objective, n_trials=20, callbacks=[trial_logger])

# --- After optimization: save the full trials dataframe and best params ---
trials_df = study.trials_dataframe()
trials_df.to_csv("bnn_optuna_full_trials.csv", index=False)

best_trial = study.best_trial
best_params = {
    "value": best_trial.value,
    "params": best_trial.params,
    "trial_number": best_trial.number
}
with open("bnn_optuna_best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

# Print summary
print("\n================= Optuna Results =================")
print(f"Best Objective: {best_trial.value:.6f}")
print("Best Hyperparameters:")
for key, value in best_trial.params.items():
    print(f"  {key:15s}: {value}")
print("Database (SQLite):", storage)
print("Per-trial CSV appended to:", csv_logfile)
print("Full trial table:", "bnn_optuna_full_trials.csv")
print("Best params JSON:", "bnn_optuna_best_params.json")
print("==================================================\n")


[I 2025-11-04 04:54:26,644] A new study created in RDB with name: bnn_study


Epoch 0 | Loss = 104969.5861
Epoch 100 | Loss = 108090.3608
Epoch 200 | Loss = 78910.2435
Epoch 300 | Loss = 65815.1371
Epoch 400 | Loss = 62707.3318
Epoch 500 | Loss = 67813.7934
Epoch 600 | Loss = 65157.8904
Epoch 700 | Loss = 75206.4724
Epoch 800 | Loss = 58716.9282


[I 2025-11-04 05:02:49,249] Trial 0 finished with value: 106.8368980884552 and parameters: {'hidden_size': 34, 'lr': 5.569202536422576e-05, 'prior_scale': 0.3451787029826713, 'dropout': 0.1467844292316441, 'expected_sigma': 0.5298495407086908}. Best is trial 0 with value: 106.8368980884552.


Epoch 0 | Loss = 90873.3360
Epoch 100 | Loss = 71508.3476
Epoch 200 | Loss = 67797.9142
Epoch 300 | Loss = 66457.3826
Epoch 400 | Loss = 39184.2513
Epoch 500 | Loss = 36682.7752
Epoch 600 | Loss = 41442.7126
Epoch 700 | Loss = 36168.6137
Epoch 800 | Loss = 32950.3524


[I 2025-11-04 05:07:22,243] Trial 1 finished with value: 697.8327506780624 and parameters: {'hidden_size': 32, 'lr': 0.0001712461708427881, 'prior_scale': 0.1616754348476353, 'dropout': 0.3152560607580664, 'expected_sigma': 0.5576706289618342}. Best is trial 0 with value: 106.8368980884552.


...

Epoch 0 | Loss = 187361.4887
Epoch 100 | Loss = 147977.8186
Epoch 200 | Loss = 147578.5534
Epoch 300 | Loss = 82916.6086
Epoch 400 | Loss = 88290.3489
Epoch 500 | Loss = 117108.0765
Epoch 600 | Loss = 93778.6957
Epoch 700 | Loss = 108416.5044
Epoch 800 | Loss = 72294.1959


[I 2025-11-04 08:46:02,671] Trial 18 finished with value: 7.690236866474152 and parameters: {'hidden_size': 41, 'lr': 3.594670271793538e-05, 'prior_scale': 0.17654440014558334, 'dropout': 0.13998205014888354, 'expected_sigma': 0.5476430353133406}. Best is trial 11 with value: 1.3618965148925781.


Epoch 0 | Loss = 79326.9818
Epoch 100 | Loss = 89846.1116
Epoch 200 | Loss = 60097.2519
Epoch 300 | Loss = 56161.5380
Epoch 400 | Loss = 37611.8912
Epoch 500 | Loss = 48509.0146
Epoch 600 | Loss = 41121.3027
Epoch 700 | Loss = 41301.2392
Epoch 800 | Loss = 51252.0152


[I 2025-11-04 09:01:17,009] Trial 19 finished with value: 1968.9386150836945 and parameters: {'hidden_size': 51, 'lr': 0.00013392321282362806, 'prior_scale': 0.11909948120232035, 'dropout': 0.21157794655950696, 'expected_sigma': 0.5175959582082337}. Best is trial 11 with value: 1.3618965148925781.



================= Optuna Results =================
Best Objective: 1.361897
Best Hyperparameters:
  hidden_size    : 45
  lr             : 0.0009975655780288084
  prior_scale    : 0.05498931322873022
  dropout        : 0.18080697430947068
  expected_sigma : 0.4692797466222822
Database (SQLite): sqlite:///bnn_optuna.db
Per-trial CSV appended to: bnn_optuna_trials.csv
Full trial table: bnn_optuna_full_trials.csv
Best params JSON: bnn_optuna_best_params.json



In [12]:
best_params = study.best_trial.params
print("Best params:", best_params)

best_trainer = BNNTrainer(
    input_dim=X_train_tensor.shape[1],
    **best_params  # cleaner unpacking
)

# Combine train + val
X_final = torch.tensor(
    np.vstack([X_train_scaled, X_val_scaled]), dtype=torch.float32
)
Y_final = torch.tensor(
    np.vstack([Y_train_scaled, Y_val_scaled]), dtype=torch.float32
)

# Retrain final model
best_trainer.train(X_final, Y_final.squeeze(), num_epochs=2000)
torch.save(best_trainer.model.state_dict(), "best_bnn_model.pth")

import json
with open("best_bnn_params.json", "w") as f:
    json.dump(best_params, f)


Best params: {'hidden_size': 45, 'lr': 0.0009975655780288084, 'prior_scale': 0.05498931322873022, 'dropout': 0.18080697430947068, 'expected_sigma': 0.4692797466222822}
Epoch 0 | Loss = 108355.5139
Epoch 100 | Loss = 70109.1259
Epoch 200 | Loss = 51156.7314
Epoch 300 | Loss = 48719.3513
Epoch 400 | Loss = 51270.2648
Epoch 500 | Loss = 45155.9023
Epoch 600 | Loss = 35990.3168
Epoch 700 | Loss = 29999.2379
Epoch 800 | Loss = 28383.7700
Epoch 900 | Loss = 31264.4900
Epoch 1000 | Loss = 24164.4334
Epoch 1100 | Loss = 23698.2404
Epoch 1200 | Loss = 21220.0616
Epoch 1300 | Loss = 19318.7537
Epoch 1400 | Loss = 23652.7900
Epoch 1500 | Loss = 18354.4395
Epoch 1600 | Loss = 16254.3569
Epoch 1700 | Loss = 16640.6473
Epoch 1800 | Loss = 13941.5705
Epoch 1900 | Loss = 15172.0053


In [13]:
import os
import zipfile
import os
import joblib
import numpy as np
import pyro

OUT = "/kaggle/working"   # Kaggle working dir

# --- 1a. Scalers (you used `scaler` and `scaler_y` in your preprocessing) ---
joblib.dump(scaler, os.path.join(OUT, "scaler_X.pkl"))
joblib.dump(scaler_y, os.path.join(OUT, "scaler_y.pkl"))
print("Saved scalers:", os.path.join(OUT, "scaler_X.pkl"), os.path.join(OUT, "scaler_y.pkl"))

# --- 1b. Scaled test arrays (you have X_test_scaled and Y_test_scaled from your preprocessing) ---
np.save(os.path.join(OUT, "X_test_scaled.npy"), X_test_scaled)
np.save(os.path.join(OUT, "Y_test_scaled.npy"), Y_test_scaled)
print("Saved test arrays:", os.path.join(OUT, "X_test_scaled.npy"), os.path.join(OUT, "Y_test_scaled.npy"))

# --- 1c. Pyro guide / param-store (trained guide params) ---
# If you used `pyro.get_param_store().save(...)` elsewhere, running again will overwrite the file but that's fine.
pyro.get_param_store().save(os.path.join(OUT, "guide_params.pt"))
print("Saved Pyro param-store:", os.path.join(OUT, "guide_params.pt"))

# --- Optional: also save trainer.model.state_dict() if you have best_trainer available ---
try:
    import torch
    torch.save(best_trainer.model.state_dict(), os.path.join(OUT, "model_state_dict.pt"))
    print("Saved model state dict:", os.path.join(OUT, "model_state_dict.pt"))
except Exception as e:
    print("Skipped saving model_state_dict (best_trainer not found or error):", e)


# Folder where Kaggle saves output files
OUTPUT_DIR = "/kaggle/working"
ZIP_NAME = "bnn_artifacts.zip"

# --- List of files to include ---
files_to_save = [
    "guide_params.pt",            # Pyro guide parameters
    "scaler_X.pkl", "scaler_y.pkl",   # Scalers
    "X_test_scaled.npy", "Y_test_scaled.npy",   # Scaled test data
    "bnn_optuna.db",              # Optuna study database
    "bnn_optuna_trials.csv",      # Per-trial logs
    "bnn_optuna_full_trials.csv", # Full trials summary
    "bnn_optuna_best_params.json" # Best params summary
]

# --- Create ZIP file ---
zip_path = os.path.join(OUTPUT_DIR, ZIP_NAME)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_save:
        file_path = os.path.join(OUTPUT_DIR, file)
        if os.path.exists(file_path):
            zipf.write(file_path, arcname=file)
            print(f"✅ Added: {file}")
        else:
            print(f"⚠️ Skipped (not found): {file}")

print(f"\n📦 All available artifacts zipped into: {ZIP_NAME}")

Saved scalers: /kaggle/working/scaler_X.pkl /kaggle/working/scaler_y.pkl
Saved test arrays: /kaggle/working/X_test_scaled.npy /kaggle/working/Y_test_scaled.npy
Saved Pyro param-store: /kaggle/working/guide_params.pt
Saved model state dict: /kaggle/working/model_state_dict.pt
✅ Added: guide_params.pt
✅ Added: scaler_X.pkl
✅ Added: scaler_y.pkl
✅ Added: X_test_scaled.npy
✅ Added: Y_test_scaled.npy
✅ Added: bnn_optuna.db
✅ Added: bnn_optuna_trials.csv
✅ Added: bnn_optuna_full_trials.csv
✅ Added: bnn_optuna_best_params.json

📦 All available artifacts zipped into: bnn_artifacts.zip


In [14]:
"""from pyro.infer import Predictive

predictive = Predictive(
    best_trainer.model_fn,
    guide=best_trainer.guide,
    num_samples=300,
)

posterior_samples = predictive(X_test_tensor)
predicted_y_samples = posterior_samples["obs"]  # shape: (1000, N)

# Mean and std
mean_prediction = predicted_y_samples.mean(dim=0)
std_prediction = predicted_y_samples.std(dim=0)

# Convert back to numpy
mean_prediction_np = mean_prediction.detach().cpu().numpy().reshape(-1, 1)
std_prediction_np = std_prediction.detach().cpu().numpy().reshape(-1, 1)
true_y_np = Y_test_tensor.detach().cpu().numpy().reshape(-1, 1)

# Inverse transform to original scale
predicted_original_scale = scaler_y.inverse_transform(mean_prediction_np)
std_original_scale = std_prediction_np * scaler_y.scale_[0]  # ✅ correct rescaling
true_original_scale = scaler_y.inverse_transform(true_y_np)
"""

'from pyro.infer import Predictive\n\npredictive = Predictive(\n    best_trainer.model_fn,\n    guide=best_trainer.guide,\n    num_samples=300,\n)\n\nposterior_samples = predictive(X_test_tensor)\npredicted_y_samples = posterior_samples["obs"]  # shape: (1000, N)\n\n# Mean and std\nmean_prediction = predicted_y_samples.mean(dim=0)\nstd_prediction = predicted_y_samples.std(dim=0)\n\n# Convert back to numpy\nmean_prediction_np = mean_prediction.detach().cpu().numpy().reshape(-1, 1)\nstd_prediction_np = std_prediction.detach().cpu().numpy().reshape(-1, 1)\ntrue_y_np = Y_test_tensor.detach().cpu().numpy().reshape(-1, 1)\n\n# Inverse transform to original scale\npredicted_original_scale = scaler_y.inverse_transform(mean_prediction_np)\nstd_original_scale = std_prediction_np * scaler_y.scale_[0]  # ✅ correct rescaling\ntrue_original_scale = scaler_y.inverse_transform(true_y_np)\n'

import numpy as np

# predicted_y_samples: shape (num_samples, num_test_points)
predicted_y_samples = posterior_samples["obs"]  # From Predictive

# 1σ (68%) interval
mean_prediction = predicted_y_samples.mean(dim=0).detach().cpu().numpy()
lower1 = predicted_y_samples.quantile(0.16, dim=0).detach().cpu().numpy()
upper1 = predicted_y_samples.quantile(0.84, dim=0).detach().cpu().numpy()

# 3σ (99.7%) interval
lower3 = predicted_y_samples.quantile(0.0015, dim=0).detach().cpu().numpy()
upper3 = predicted_y_samples.quantile(0.9985, dim=0).detach().cpu().numpy()

# Inverse transform (to original scale)
mean_pred_orig = scaler_y.inverse_transform(mean_prediction.reshape(-1, 1)).flatten()
lower1_orig = scaler_y.inverse_transform(lower1.reshape(-1, 1)).flatten()
upper1_orig = scaler_y.inverse_transform(upper1.reshape(-1, 1)).flatten()
lower3_orig = scaler_y.inverse_transform(lower3.reshape(-1, 1)).flatten()
upper3_orig = scaler_y.inverse_transform(upper3.reshape(-1, 1)).flatten()

# Ground truth inverse transformed
true_y_orig = scaler_y.inverse_transform(Y_test_tensor.cpu().numpy().reshape(-1, 1)).flatten()



import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(8, 6))

# Sort by true Y (chirp mass) for smooth fill_between
sorted_indices = np.argsort(true_y_orig.flatten())
x_sorted = true_y_orig[sorted_indices]
mean_sorted = mean_pred_orig[sorted_indices]
lower1_sorted = lower1_orig[sorted_indices]
upper1_sorted = upper1_orig[sorted_indices]
lower3_sorted = lower3_orig[sorted_indices]
upper3_sorted = upper3_orig[sorted_indices]

# Plot 3σ fill (lightest)
plt.fill_between(
    x_sorted,
    lower3_sorted,
    upper3_sorted,
    color='lightblue',
    alpha=0.3,
    label='3σ interval'
)

# Plot 1σ fill (darker)
plt.fill_between(
    x_sorted,
    lower1_sorted,
    upper1_sorted,
    color='royalblue',
    alpha=0.4,
    label='1σ interval'
)

# Mean predictions (scatter)
plt.scatter(true_y_orig, mean_pred_orig, alpha=0.5, color='darkorange', label='Predicted Mean')

# Ideal y = x line
plt.plot([true_y_orig.min(), true_y_orig.max()],
         [true_y_orig.min(), true_y_orig.max()],
         'k--', label="Ideal")

# Plot settings
plt.xlabel('True Chirp Mass')
plt.ylabel('Predicted Chirp Mass')
plt.title('Predicted vs True Chirp Mass with 1σ and 3σ Intervals')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


x_axis = np.arange(len(true_y_orig))
plt.fill_between(x_axis, lower3_orig, upper3_orig, color='lightblue', alpha=0.3, label='3σ')
plt.fill_between(x_axis, lower1_orig, upper1_orig, color='royalblue', alpha=0.4, label='1σ')
plt.plot(x_axis, mean_pred_orig, 'darkorange', label='Predicted Mean')
plt.plot(x_axis, true_y_orig, 'k--', label='True')

print("Predicted:", predicted_original_scale[:10])
print("True:", true_original_scale[:10])

plt.figure(figsize=(8, 5))
plt.scatter(Y_data, SNRs, color='darkblue', alpha=0.5, label='SNR values')
plt.title('SNR vs Chirp Mass')
plt.xlabel('Chirp Mass')
plt.ylabel('SNR')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Quantitative analysis of the prediction



import numpy as np

def bnn_metrics(y_true, y_pred_mean, y_pred_std):
    # Flatten arrays in case they're shaped (N,1)
    y_true = y_true.flatten()
    y_pred_mean = y_pred_mean.flatten()
    y_pred_std = y_pred_std.flatten()

    # ----- Prediction Accuracy -----
    rmse = np.sqrt(np.mean((y_pred_mean - y_true)**2))
    mae = np.mean(np.abs(y_pred_mean - y_true))
    
    # Percent versions (relative to mean true value)
    mean_true = np.mean(y_true)
    rmse_pct = (rmse / mean_true) * 100
    mae_pct = (mae / mean_true) * 100
    
    # ----- Bias -----
    bias = np.mean(y_pred_mean - y_true)
    bias_pct = (bias / mean_true) * 100
    
    # ----- Uncertainty Calibration -----
    within_1sigma = np.mean((y_true >= y_pred_mean - y_pred_std) & 
                            (y_true <= y_pred_mean + y_pred_std))
    within_3sigma = np.mean((y_true >= y_pred_mean - 3*y_pred_std) & 
                            (y_true <= y_pred_mean + 3*y_pred_std))
    
    # Average interval widths
    avg_width_1sigma = np.mean(2 * y_pred_std)
    avg_width_3sigma = np.mean(6 * y_pred_std)
    
    # Store results
    results = {
        "RMSE": rmse,
        "RMSE_%": rmse_pct,
        "MAE": mae,
        "MAE_%": mae_pct,
        "Bias": bias,
        "Bias_%": bias_pct,
        "Coverage_1σ": within_1sigma,
        "Coverage_3σ": within_3sigma,
        "AvgWidth_1σ": avg_width_1sigma,
        "AvgWidth_3σ": avg_width_3sigma
    }
    
    # Pretty print
    print("\nBNN Performance Metrics")
    print("="*40)
    print(f"RMSE          : {rmse:,.2f} ({rmse_pct:.2f}%)")
    print(f"MAE           : {mae:,.2f} ({mae_pct:.2f}%)")
    print(f"Bias          : {bias:,.2f} ({bias_pct:.2f}%)")
    print(f"Coverage (±1σ): {within_1sigma*100:.1f}%")
    print(f"Coverage (±3σ): {within_3sigma*100:.1f}%")
    print(f"Avg Width ±1σ : {avg_width_1sigma:,.2e}")
    print(f"Avg Width ±3σ : {avg_width_3sigma:,.2e}")
    print("="*40)
    
    return results

metrics = bnn_metrics(true_original_scale, predicted_original_scale, std_original_scale)
